In [1]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [3]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("../data/kg_train.csv", encoding='latin-1')

# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

df =data.copy()
pd.set_option('display.max_colwidth',100)
df.head()

(1000, 2)


,text,label
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL I AM MIKE CHUKWU , THE MANAGER, BILLS AND EXCHANG...",1
1,Will do.,0
2,Nora--Cheryl has emailed dozens of memos about Haiti to me this weekend. Can you please print th...,0
3,Dear Sir=2FMadam=2C I know that this proposal might be a surprise to you but it as an emergency=...,1
4,fyi,0


### Let's divide the training and test set into two partitions

In [4]:
# Your code
from sklearn.model_selection import train_test_split
X = df.drop(columns='label')
y = df.label

# X_train, X_test, y_train, y_test = train_test_split(
#     X, y,
#     test_size=0.2,
#     random_state=42,
#     stratify=y
# )

# print("X_train:", X_train.shape)
# print("X_test :", X_test.shape)
# print("y_train:", y_train.shape)
# print("y_test :", y_test.shape)

## Data Preprocessing

In [5]:
import string, nltk
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])

from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [6]:
# Your code
import re

def clean_html(text):

    # Remove script and style content
    text = re.sub(r'<(script|style).*?>.*?</\1>', '', text, flags=re.DOTALL | re.IGNORECASE)
    # Remove HTML comments
    text = re.sub(r'<!--.*?-->', '', text, flags=re.DOTALL)
    # Remove remaining HTML tags
    text = re.sub(r'<.*?>', '', text)
    return text

X["text"] = X['text'].apply(clean_html)

In [7]:
X['text'].head()

0    DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL I AM MIKE CHUKWU , THE MANAGER, BILLS AND EXCHANG...
1                                                                                               Will do.
2    Nora--Cheryl has emailed dozens of memos about Haiti to me this weekend. Can you please print th...
3    Dear Sir=2FMadam=2C I know that this proposal might be a surprise to you but it as an emergency=...
4                                                                                                    fyi
Name: text, dtype: str

In [8]:
text1=X['text'][0]
text1

"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL I AM MIKE CHUKWU , THE MANAGER, BILLS AND EXCHANGE AT THE FOREIGN REMITTANCE DEPARTMENT OF THE ZENITH INTERNATIONAL BANK PLC. I AM WRITING THIS LETTER TO ASK FOR YOUR SUPPORT AND COOPERATION TO CARRY OUT THIS BUSINESS OPPORTUNITY IN MY DEPARTMENT. WE DISCOVERED AN ABANDONED SUM OF $15,000,000.00 (FIFTEEN MILLION UNITED STATES DOLLARS ONLY) IN AN ACCOUNT THAT BELONGS TO ONE OF OUR FOREIGN CUSTOMERS WHO DIED ALONG WITH HIS ENTIRE FAMILY OF A WIFE AND TWO CHILDREN IN NOVEMBER 1997 IN A PLANE CRASH. SINCE WE HEARD OF HIS DEATH, WE HAVE BEEN EXPECTING HIS NEXT-OF-KIN TO COME OVER AND PUT CLAIMS FOR HIS MONEY AS THE HEIR,BECAUSE WE CANNOT RELEASE THE FUND FROM HIS ACCOUNT UNLESS SOMEONE APPLIES FOR CLAIM AS THE NEXT-OF-KIN TO THE DECEASED AS INDICATED IN OUR BANKING GUIDELINES. UNFORTUNATELY, NEITHER THEIR FAMILY MEMBER NOR DISTANT RELATIVE HAS EVER APPEARED TO CLAIM THE SAID FUND. UPON THIS DISCOVERY,I AND OTHER OFFICIALS IN MY DEPARTMENT HAVE

- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [9]:
# Your code

def clean_text(text):
    
    # Convert to lowercase
    text = text.lower()

    # Remove prefixed b (byte strings)
    text = re.sub(r"^b\s+", "", text)
        
    # Remove numbers
    text = re.sub(r"\d+", "", text)
    
    # Remove special characters (keep only letters and spaces)
    text = re.sub(r"[^a-z\s]", "", text)
    
    # Remove single characters
    text = re.sub(r"\b[a-z]\b", "", text)
    
    # Remove single characters from start
    text = re.sub(r"^[a-z]\s+", "", text)
    
    # Substitute multiple spaces with single space
    text = re.sub(r"\s+", " ", text)
    
    return text.strip()

In [10]:
# Test this fucntion of single string
text1 = clean_text(text1)
text1

'dear sir strictly private business proposal am mike chukwu the manager bills and exchange at the foreign remittance department of the zenith international bank plc am writing this letter to ask for your support and cooperation to carry out this business opportunity in my department we discovered an abandoned sum of fifteen million united states dollars only in an account that belongs to one of our foreign customers who died along with his entire family of wife and two children in november in plane crash since we heard of his death we have been expecting his nextofkin to come over and put claims for his money as the heirbecause we cannot release the fund from his account unless someone applies for claim as the nextofkin to the deceased as indicated in our banking guidelines unfortunately neither their family member nor distant relative has ever appeared to claim the said fund upon this discoveryi and other officials in my department have agreed to make business with you and release the

In [11]:
X['text'] = X['text'].apply(clean_text)

X['text']

0      dear sir strictly private business proposal am mike chukwu the manager bills and exchange at the...
1                                                                                                  will do
2      noracheryl has emailed dozens of memos about haiti to me this weekend can you please print them ...
3      dear sirfmadamc know that this proposal might be surprise to you but it as an emergencyein nutsh...
4                                                                                                      fyi
                                                      ...                                                 
995    so whats the latest it sounds contradictory and when does af have to decide shall we take the or...
996    transfer of million pounds to youraccountmy name is mrejis don and work in theinternational oper...
997                                                  barb will call to explain are you back in the country
998                                  

## Now let's work on removing stopwords
Remove the stopwords.

In [12]:
len(text1)

2195

In [13]:
# Your code

# First tokenize the words and apply stop words
from nltk import word_tokenize
from nltk.corpus import stopwords
stop_words_set = set(stopwords.words('english'))

def remove_stopwords(text):

    tokens = word_tokenize(text)
 
    filter_tokens = [word for word in tokens if word not in stop_words_set]

    return " ".join(filter_tokens)

In [14]:
print("Before stop words:\n", X['text'][0][:200])
X["text"] = X["text"].apply(remove_stopwords)

print("After stop words:\n", X['text'][0][:200])

Before stop words:
 dear sir strictly private business proposal am mike chukwu the manager bills and exchange at the foreign remittance department of the zenith international bank plc am writing this letter to ask for yo
After stop words:
 dear sir strictly private business proposal mike chukwu manager bills exchange foreign remittance department zenith international bank plc writing letter ask support cooperation carry business opportu


## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [15]:
len(text1)

2195

In [16]:
text1 = remove_stopwords(text1) 
print(text1)

dear sir strictly private business proposal mike chukwu manager bills exchange foreign remittance department zenith international bank plc writing letter ask support cooperation carry business opportunity department discovered abandoned sum fifteen million united states dollars account belongs one foreign customers died along entire family wife two children november plane crash since heard death expecting nextofkin come put claims money heirbecause release fund account unless someone applies claim nextofkin deceased indicated banking guidelines unfortunately neither family member distant relative ever appeared claim said fund upon discoveryi officials department agreed make business release total amount account heir fund since one came discovered maintained account bank otherwise fund returned banks treasury unclaimed fund agreed ratio sharing stated thus foreign partner us officials department settlement local foreign expences incurred us course business upon successful completion tra

In [17]:
len(text1)

1519

In [18]:
new_sentence = text1

In [19]:
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(tag):
    #Maps it to a WordNet-compatible tag
    tag_dict = {"J": wordnet.ADJ,
                "N": wordnet.NOUN,
                "V": wordnet.VERB,
                "R": wordnet.ADV
    }
    return tag_dict.get(tag[0], wordnet.NOUN) # returns the word type (Noun if we have not found)

def lemmatize_tokens(text):

    tokens = word_tokenize(text)

    tagged_tokens = nltk.pos_tag(tokens)

    lemmatize_words= [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in tagged_tokens]

    return lemmatize_words

In [20]:
print("Before lemmatize:\n", X['text'][0][:100])
X["text"] = X["text"].apply(lemmatize_tokens)
print("After lemmatize:\n", X['text'][0][:50])

Before lemmatize:
 dear sir strictly private business proposal mike chukwu manager bills exchange foreign remittance de
After lemmatize:
 ['dear', 'sir', 'strictly', 'private', 'business', 'proposal', 'mike', 'chukwu', 'manager', 'bill', 'exchange', 'foreign', 'remittance', 'department', 'zenith', 'international', 'bank', 'plc', 'write', 'letter', 'ask', 'support', 'cooperation', 'carry', 'business', 'opportunity', 'department', 'discover', 'abandon', 'sum', 'fifteen', 'million', 'united', 'state', 'dollar', 'account', 'belong', 'one', 'foreign', 'customer', 'die', 'along', 'entire', 'family', 'wife', 'two', 'child', 'november', 'plane', 'crash']


## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [22]:
# Apply train-test split to prevent data leak 
X_train, X_test, y_train, y_test= train_test_split(
    X, y, 
    test_size= 0.2 , 
    random_state=42,
    stratify=y
)


In [28]:
X_train.head()

,text
442,"[dearc, good, day, hope, finecdear, writting, mail, due, respect, heartful, tear, since, know, m..."
962,"[mr, henry, kaborethe, chief, auditor, inchargeforeign, remittance, unitafrican, development, ba..."
971,[]
190,"[desk, dradamu, ismalerauditing, account, managerbank, africa, boaouagadougouburkina, fasobank, ..."
551,"[dear, friend, name, loi, cestradathe, wife, mr, josephestrada, former, president, philippine, l..."


In [32]:
X_train_text = X_train["text"].apply(lambda x: " ".join(x))
X_test_text  = X_test["text"].apply(lambda x: " ".join(x))

X_train_text

442    dearc good day hope finecdear writting mail due respect heartful tear since know meet previously...
962    mr henry kaborethe chief auditor inchargeforeign remittance unitafrican development bankadbouaga...
971                                                                                                       
190    desk dradamu ismalerauditing account managerbank africa boaouagadougouburkina fasobank websiteww...
551    dear friend name loi cestradathe wife mr josephestrada former president philippine locatedin sou...
                                                      ...                                                 
625    httpwwwrteieewszimbabwehtml get yourcontact search internethere south africa know person iam rel...
53     ivdearnbspit pleasure write much consideration since telephonenbsp communication suitable enough...
728                               think good shape immelt indra huntsman make call mark penn terry helpful
664    permission change statement sp

In [37]:
# Your code

from sklearn.feature_extraction.text import CountVectorizer

def get_bag_of_words(X_train, X_test):

    vectorizer = CountVectorizer(max_features=20)
    
    X_train_bow = vectorizer.fit_transform(X_train_text)
    X_test_bow  = vectorizer.transform(X_test_text)
    
    return X_train_bow, X_test_bow, vectorizer

In [38]:
X_train_bow, X_test_bow, vectorizer = get_bag_of_words(X_train, X_test)

In [44]:
print("Number of features:", len(vectorizer.get_feature_names_out()))
print("First 20 features:", vectorizer.get_feature_names_out()[:10])

Number of features: 20
First 20 features: ['account' 'also' 'bank' 'business' 'company' 'contact' 'country'
 'deposit' 'foreign' 'fund']


In [45]:
print("Number of features:", len(vectorizer.get_feature_names_out()))
print("First 20 features:", vectorizer.get_feature_names_out()[:5])

Number of features: 20
First 20 features: ['account' 'also' 'bank' 'business' 'company']


## Extra features

In [52]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

X_train['money_mark'] = X_train['text'].str.contains(money_simbol_list)*1
X_train['suspicious_words'] = X_train['text'].str.contains(suspicious_words)*1
X_train['text_len'] = X_train['text'].apply(lambda x: len(x)) 

X_test['money_mark'] = X_test['text'].str.contains(money_simbol_list)*1
X_test['suspicious_words'] = X_test['text'].str.contains(suspicious_words)*1
X_test['text_len'] = X_test['text'].apply(lambda x: len(x)) 

X_train.head()

,text,money_mark,suspicious_words,text_len
442,"[dearc, good, day, hope, finecdear, writting, mail, due, respect, heartful, tear, since, know, m...",NaN,NaN,144
962,"[mr, henry, kaborethe, chief, auditor, inchargeforeign, remittance, unitafrican, development, ba...",NaN,NaN,244
971,[],NaN,NaN,0
190,"[desk, dradamu, ismalerauditing, account, managerbank, africa, boaouagadougouburkina, fasobank, ...",NaN,NaN,47
551,"[dear, friend, name, loi, cestradathe, wife, mr, josephestrada, former, president, philippine, l...",NaN,NaN,177


In [57]:
X_train_new_text = X_train["text"].apply(lambda x: " ".join(x))
X_test_new_text = X_test["text"].apply(lambda x: " ".join(x))

## How would work the Bag of Words with Count Vectorizer concept?

In [58]:
# Your code

# Fit only on training text
X_train_bow = vectorizer.fit_transform(X_train_new_text)

# Transform test text
X_test_bow = vectorizer.transform(X_test_new_text)

# Show vocabulary
print("Vocabulary:")
print(vectorizer.get_feature_names_out())

# Show document-term matrix
print("\nDocument-Term Matrix (Train):")
print(X_train_bow.toarray())

print("\nTrain shape:", X_train_bow.shape)
print("Test shape:", X_test_bow.shape)

Vocabulary:
['account' 'also' 'bank' 'business' 'company' 'contact' 'country'
 'deposit' 'foreign' 'fund' 'get' 'know' 'make' 'million' 'money' 'next'
 'one' 'transaction' 'transfer' 'want']

Document-Term Matrix (Train):
[[0 3 0 ... 0 0 2]
 [4 0 7 ... 4 5 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [1 1 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]

Train shape: (800, 20)
Test shape: (200, 20)


## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [60]:
# Your code


from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(X_test_new_text)
feature_names = vectorizer.get_feature_names_out()
tfidf_array = tfidf_matrix.toarray()

for i, doc_tfidf in enumerate(tfidf_array):
    print(f"Document {i+1}:")
    for j, score in enumerate(doc_tfidf):
        if score > 0:
            print(f"  {feature_names[j]}: {score:.4f}")
    print()

Document 1:
  acknowledge: 0.1183
  also: 0.0672
  andutilizing: 0.1414
  ascertain: 0.1414
  bank: 0.0603
  banker: 0.1312
  bankmr: 0.1414
  business: 0.0603
  classified: 0.1414
  confidentiality: 0.0875
  contact: 0.0583
  contactme: 0.1312
  country: 0.0588
  customer: 0.2197
  deal: 0.0906
  decease: 0.1969
  disposition: 0.1414
  explanationsyou: 0.1414
  full: 0.0749
  furtherdiscussions: 0.1414
  great: 0.1969
  guarantee: 0.1240
  humble: 0.1312
  individual: 0.0985
  information: 0.1447
  interest: 0.0834
  introduce: 0.0924
  joyce: 0.1414
  kinto: 0.1414
  legal: 0.1035
  legally: 0.1240
  maintain: 0.0985
  matter: 0.0834
  may: 0.0749
  mitchell: 0.1414
  mr: 0.1371
  must: 0.0906
  name: 0.0723
  need: 0.0693
  next: 0.0693
  note: 0.1035
  number: 0.0732
  origin: 0.1414
  partner: 0.0740
  personalpermit: 0.1414
  please: 0.0700
  possess: 0.1414
  present: 0.0810
  profession: 0.1137
  receive: 0.0618
  relative: 0.1009
  right: 0.1009
  self: 0.1183
  share: 0.0686


In [61]:
tfidf_array.shape

(200, 5329)

## And the Train a Classifier?

In [ ]:
# Your code

### Extra Task - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

The classifier can not be changed!!! It must be the MultinimialNB with default parameters!

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [ ]:
# Your code